#MATRICES DE TF, IDF Y TF-IDF

Carlos Francisco Montoya Urzúa

In [17]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

#CORPUS
corpus = [
    'el coche es rapido',
    'el coche es azul',
    'un coche rapido azul'
]

vectorizador = CountVectorizer()
matriz_dispersa = vectorizador.fit_transform(corpus)

df_unigramas = pd.DataFrame(
    matriz_dispersa.toarray(),
    columns=vectorizador.get_feature_names_out()
)
df_unigramas.index = ['Documento 1', 'Documento 2', 'Documento 3']

print("CALCULO DE TF: Matriz de Unigramas")
display(df_unigramas)


CALCULO DE TF: Matriz de Unigramas


,azul,coche,el,es,rapido,un
Documento 1,0,1,1,1,1,0
Documento 2,1,1,1,1,0,0
Documento 3,1,1,0,0,1,1


In [18]:
import pandas as pd
import numpy as np

def calcular_tf(corpus):
    # Obtener el vocabulario único preservando el orden en que aparecen por primera vez
    vocabulario = []
    for doc in corpus:
        for palabra in doc.split():
            if palabra not in vocabulario:
                vocabulario.append(palabra)
    vocabulario = sorted(vocabulario)

    # Calcular TF: (frecuencia de término / total de términos en el documento)
    filas_tf = []
    for doc in corpus:
        palabras_doc = doc.split()
        total_palabras = len(palabras_doc)

        # Contar ocurrencias de cada palabra del vocabulario en este documento
        frecuencias = {palabra: palabras_doc.count(palabra) / total_palabras for palabra in vocabulario}
        #frecuencias = {palabra: palabras_doc.count(palabra) for palabra in vocabulario}
        filas_tf.append(frecuencias)

    # Crear el DataFrame
    df_tf = pd.DataFrame(filas_tf)
    df_tf.index = [f'Documento {i+1}' for i in range(len(corpus))]
    return df_tf

# Generar y mostrar la tabla de frecuencias TF
df_frecuencias_tf = calcular_tf(corpus)
print("MATRIZ DE TF (Term Frequency):")
display(df_frecuencias_tf.T)

MATRIZ DE TF (Term Frequency):


,Documento 1,Documento 2,Documento 3
azul,0.00,0.25,0.25
coche,0.25,0.25,0.25
el,0.25,0.25,0.00
es,0.25,0.25,0.00
rapido,0.25,0.00,0.25
un,0.00,0.00,0.25


In [19]:
from math import log10
import pandas as pd

def document_frecuency(t, corpus):
    df = 0
    for doc in corpus:
        if t in doc.split():
            df += 1
    return df

def IDF(t, corpus):
      idf = log10(len(corpus)/document_frecuency(t, corpus))
      return idf


# Obtener la lista de palabras únicas del corpus
vocabulario = set()
for doc in corpus:
    vocabulario.update(doc.split())
vocabulario = sorted(list(vocabulario))

# Crear los datos para el DataFrame
datos = []
for palabra in vocabulario:
    df_val = document_frecuency(palabra, corpus)
    idf_val = IDF(palabra, corpus)
    datos.append({
        'Palabra': palabra,
        'DF': df_val,
        'IDF': idf_val
    })

# Crear y mostrar el DataFrame
df_idf = pd.DataFrame(datos)
print("CALCULO DE IDF")
display(df_idf)

CALCULO DE IDF


,Palabra,DF,IDF
0,azul,2,0.176091
1,coche,3,0.000000
2,el,2,0.176091
3,es,2,0.176091
4,rapido,2,0.176091
5,un,1,0.477121


In [20]:
# Asegurar que df_idf tenga la columna 'Palabra' como índice para facilitar la alineación
df_idf_indexed = df_idf.set_index('Palabra')

# Obtener la matriz TF transpuesta
df_tf_transpuesta = df_frecuencias_tf.T
#display(df_tf_transpuesta)

# Multiplicar el TF de cada palabra en cada documento por su IDF
# pandas alinea automáticamente las filas al multiplicar un DataFrame por una Serie (el IDF)
df_tfidf = df_tf_transpuesta.multiply(df_idf_indexed['IDF'], axis=0)

print("MATRIZ TF-IDF:")
display(df_tfidf)

MATRIZ TF-IDF:


,Documento 1,Documento 2,Documento 3
azul,0.000000,0.044023,0.044023
coche,0.000000,0.000000,0.000000
el,0.044023,0.044023,0.000000
es,0.044023,0.044023,0.000000
rapido,0.044023,0.000000,0.044023
un,0.000000,0.000000,0.119280
